# Day 7 -Advanced Delta Lake + SCD Type 2

In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_database(), current_catalog(), current_schema();

In [0]:
%sql
select * from customer360.customer_master order by custId;

In [0]:
%sql
DESCRIBE HISTORY customer360.customer_master;

## Part 2 - Update

**Suppose** customer 1005 moved to Plano:

In [0]:
%sql
update customer360.customer_master
set city = 'Plano'
where custId = 1005;

In [0]:
%sql
DESCRIBE HISTORY customer360.customer_master;

## Part 3 - DELETE


Suppose customer 1005 was created accidentally:

In [0]:
%sql
DELETE FROM CUSTOMER360.CUSTOMER_MASTER WHERE CUSTID = 1005

In [0]:
%sql
DESCRIBE HISTORY customer360.customer_master;

In [0]:
%sql
select * from customer360.customer_master;

## Part 4  - Time Travel


suppose you want to see the custimer 1005 again which was just deleted

**Read an earlier version without changing the current table.**

In [0]:
%sql
select * from customer360.customer_master version as of 30
order by custid;

## Part 5 -  RESTORE

- Suppose the DELETE was genuinely accidental and you want the table **itself returned to its previous state.**
- RESTORE itself creates another table version.

    `````
    Time Travel
        ↓
    READ old state

    RESTORE
        ↓
    make an earlier state the current table state


In [0]:
%sql
RESTORE table  customer360.customer_master to version as of 30;


In [0]:
%sql
select * from customer360.customer_master 
order by custid;

describe history customer360.customer_master;

# Part 6 - VACCUM

- VACUUM **removes old/unreferenced data files** that are no longer required by the current table and are older than the **retention threshold.**


# Part 7 — OPTIMIZE


- OPTIMIZE reorganizes/compacts data files to improve layout and query efficiency.

````
OPTIMIZE
   ↓
Improve data-file layout/performance

VACUUM
   ↓
Remove obsolete files after retention period

## Part 8 — SCD Type 1 vs Type 2

- SCD Type 1 --> values are **OVERWRITTEN**, 
```
John Moved from Plano to Frisco, old value ("Plano") is overwritten by new value("Frisco")
```
- SCD Type 2--> creates a NEW RECORD and sets the activeFlag = true and deactivates the old record with the isActive= False. 

# Senior scenario

Your customer dimension contains 500 million records. Every night you receive 5 million customer records, but only 300,000 actually changed. The business requires full history. How would you design the incremental Delta/SCD2 process without rewriting the entire target?

### Step 1- create CUSTOMER_HISTORY TABLE

In [0]:
from pyspark.sql.functions import *

scd_data = [
    (1001, "John Smith", "Active", "Dallas"),
    (1002, "Mary Jones", "Active", "Plano"),
    (1003, "Steve Brown", "Active", "McKinney")
]

scd_columns = ["custId", "custName", "status", "city"]
dfInitial = spark.createDataFrame(scd_data, scd_columns)

dfInitialSCD = (
    dfInitial
    .withColumn("startDate", current_date())
    .withColumn("endDate",lit(None).cast("date"))
    .withColumn("isCurrent",lit(True))
                )

dfInitialSCD.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360.customer_history")

In [0]:
%sql
select * from customer360.customer_history;

###  STEP 2- ADD Changes


In [0]:
from delta.tables import DeltaTable

upt_data = [
    (1001, "John Smith", "Suspended", "Dallas"),
    (1002, "Mary Jones", "Active", "Plano"),
    (1004, "Sarah Wilson", "Active", "Frisco")
]

dfUpdates = spark.createDataFrame(upt_data, scd_columns)

dfSCD2 = (
    dfUpdates
    .withColumn("startDate", current_date())
    .withColumn("endDate",lit(None).cast("date"))
    .withColumn("isCurrent",lit(True))
                )

# Get current ACTIVE records to compare for actual changes
dfCurrent = spark.table("workspace.customer360.customer_history").filter(col("isCurrent") == True)

# Only keep records that are new or have actual value changes
dfChanges = (
    dfSCD2.alias("new")
    .join(dfCurrent.alias("current"), "custId", "left")
    .filter(
        col("current.custId").isNull()
        |
        ~col("current.custName").eqNullSafe(col("new.custName"))
        |
        ~col("current.status").eqNullSafe(col("new.status"))
        |
        ~col("current.city").eqNullSafe(col("new.city"))
    )
    .select("new.*")
)

# deactive OLD record
currentMaster = DeltaTable.forName(spark,"workspace.customer360.customer_history")
(
currentMaster.alias("current").merge(
    dfChanges.alias("new"), "current.custId = new.custId AND current.isCurrent = true"
    ) 
    .whenMatchedUpdate(
        set = {
            "endDate": current_date(), "isCurrent": "false"
        }
    )
    .execute()
)

dfChanges.write.format("delta").mode("append").saveAsTable("workspace.customer360.customer_history")


In [0]:
%sql select * from customer360.customer_history order by custId, isCurrent; 

### idempotent
- An idempotent pipeline can process the same input repeatedly without producing additional unintended changes after the first successful run.

consider above example- your current implementation is idempotent for this same input.

Suppose you run it a** second time**.

After run #1, current records are:
```
1001  Suspended   current
1002  Active      current
1003  Active      current
1004  Active      current
```
On run #2, your source contains:
```
1001  Suspended
1002  Active
1004  Active
```
Your comparison finds:
```
1001 → same
1002 → same
1004 → same
```
Therefore:
```
dfChanges = empty
```
So:
```
MERGE  → updates 0 rows
APPEND → inserts 0 rows
```